## 2. Time and Windows in Flink

- We have covered how flink sets itself up to do its job. Think of it as Flink's "Body" of sorts 

- In this section, we will dive into the "Brain" of Flink; we will elaborate on how flink sees "time", and uses this to manage state across windows

### Time in Flink

- Flink recognises 3 concepts of time;
    - Event Time: timestamp attached to the data at the point of creation. This is the gold standard of the notion of time
        
    - Processing Time: Actual wall clock time of the TaskManager machine.
        - This is the fastest to access, because the Task Manager accesses it directly
        - BUT imagine a situation where the task manager is under heavy load. 
            - A data point with event time at 12:04 comes into the TM
            - The TM is still processing window from 11:55 to 12:00. 
            - Since it is under load, the 12:04 event only appears in the TM at 12:06.
            - So it appears the 12:04 event as if it is a 12:06 event, because it was relying on the time when the event entered the TM, not when the event executed!
            - This is most often used when you don't really care about the actual event order. For example, if you just want to check the throughput of the app, this is the most appropriate

    - Ingestion Time: Time that the data entered the flink source
        - This is kind of a "better" processing time; you only record the time of the event when it hits your flink app
        - But obviously this also has shortfalls
            - Let's say a nuclear reactor has an app monitoring core temperature.
            - At 1pm, core temperature hits 1000 degrees
            - But because of network issues, this doesn't hit your flink app until 130pm
            - Obviously, this is a problem, because you will record that 1000 degrees happened at 130pm, instead of 1pm
        - So even though this is "better" in the case where you care about the correctness of the event time, this is worse if you want to do something like measure the throughput in a TaskManager; if nothing enters your TM for 10 seconds, Ingestion time doesn't move forward at all, but processing time still moves forward

    - Event Time: As you can see, neither processing nor ingestion time is ideal. This is why Event Time is considered the **gold standard** of time in flink
        - Basically, instead of Flink trying to guess a time where the event arrives, you get the producer of the data to couple send the time of the event generation to flink!
        - This is, of course, the best representation of when the event actually happened

- Why is time important in Flink?
    - As we discussed, Flink's job is to do **stateful** aggregation
    - That is, we want to do stuff like "what is count of events in the last 5 minutes"?
    - So Flink's job to figure out when "last 5 minutes" has elapsed!! Basically, Flink has to know when to stop waiting for data, and just execute the code it has been given

- Flink does this via a metadata packet known as **watermarks**
    
- Let's trace how watermarks pass through Flink's internal system
    1. Flink receives data from some external system (Kafka, JSON, etc)
    2. The **Source Task** converts this data into specific Java/Scala objects. Upon receiving the data, this operator also desides what "time" you want to assign to this batch of data. 
        - How the source decides this is known as the **watermark strategy** 
        - There are 3 broad categories of watermark strategies you can use in Flink
            - Let's assume you have the following data times arriving at your source in this order: 
                - Batch 1: [10:00, 10:02, 10:05]
                - Batch 2: [10:03, 10:05, 10:07]
            - **Monotonously Increasing**: You tell flink that data will ALWAYS arrive in order. 
                - Upon receiving Batch 1, Flink advances the watermark to `10:05`. If we aggregate in 5 minute windows, this may be the point when downstream operators close the window!
                - The downside is obvious. If you have a second batch of data producing an entry at `10:03`, it gets dropped, because the window is already closed
            - **Bounded Out-of-Order**: You tell flink that data may arrive out of order, but the out of orderness will not exceed some time frame. 
                - Suppose we set our watermark strategy to this, at 2mins
                - Upon receiving Batch 1, Flink advances the watermark to `10:05 - 2min = 10:03`
                - Windows that need to close at `10:05` DO NOT close
                - Then Batch 2 arrives, and Flink advances the watermark to `10:07 - 2min = 10:05`
                - Window closes, and `10:03`  event is captured! Simply because we delayed our watermark by 2 mins
            - **Punctuated Watermark**: You pass flink a boolean to determine when a watermark should be incremented
    3. Now that the source has processed the watermark, it flows to the operators downstream
        - These operators now decide whether or not to compute a window, depending on the watermark that was determined by the source!!

### Windows in Flink

- Having covered the concept of "Time" in Flink, let's now discuss what it means to have a "Window" in Flink

- Since a stream never ends, we can't do a SUM() on everything. We have to "slice" the stream into buckets called Windows.

- What types of windowing does Flink support?
    - Tumbling: Fixed size, no overlap. 
        - [10:00, 10:05)
        - [10:05, 10:10)
        - ...
    - Sliding: Fixed size, overlapping
        - [10:00, 10:05)
        - [10:01, 10:06)
        - ...
    - Session
        - Accumulates continuously until some condition is met (e.g. log events until last event received was 30mins ago, then close window)
    - Global
        - Everything goes into a single bucket
        - Custom trigger logic required

- Flink will use the idea of "time" that we discussed in the last 2 sections to decide when to open and close a window

- Flink is intelligent enough that we can do even better than setting a "Out of Order Tolerance" when it comes to accepting late data
    - For windows, we can change a configuration known as **AllowedLateness**
    - Even if a window has already computed and returned a value, **AllowedLateness** lets flink retain the computed value in memory for some time. If a new data point enters, we can use it to recompute the data for that window, incrementaly refining the data!

### Handling the "Stragglers": Late Data & Side Outputs

- Even with a generous Watermark strategy, network partitions or producer hiccups can cause events to arrive after a window has already "fired" and its state has been cleaned up. 

- Flink provides two primary layers of defense to ensure these events aren't simply lost to the void:
    - Allowed Lateness: The "Grace Period"
        - AllowedLateness allows Flink to keep a window's summary in memory for an additional period after the Watermark has passed the window end-time
        - When a late element arrives within the "allowed lateness" period, Flink does not create a new window; instead, it triggers a late firing
        - Transformation: This incrementally updates the previous result, which is critical for accuracy in financial or billing systems
        - This consumes more managed state (RocksDB), as the "locker" for that window must remain open longer. Think of this as storing all processed windows for an extended period of time on disk
    
    - Side Outputs: The "Safety Net"
        - Once the AllowedLateness period expires, any subsequent data for that window is dropped by default. To prevent data loss, you can redirect these "too late" events into a Side Output
        - A Side Output is a secondary stream branch that captures data that doesn't fit the main pipeline's logic (like late events)
        - You can define an OutputTag to identify the late stream, which can then be written to a "Dead Letter" sink (like an S3 bucket or a specific Kafka topic) for later manual inspection or reprocessing

    ```java
        // 1. Define a tag for the late data
        final OutputTag<ClickLog> lateDataTag = new OutputTag<ClickLog>("late-clicks"){};

        SingleOutputStreamOperator<WindowResult> result = input
            .keyBy(log -> log.getUserId())
            .window(TumblingEventTimeWindows.of(Time.minutes(5)))
            .allowedLateness(Time.minutes(10)) // Keep window state for 10 extra mins
            .sideOutputLateData(lateDataTag)    // Divert anything later than 10 mins
            .aggregate(new MyAggregateFunction());

        // 2. Access the late stream to save it
        DataStream<ClickLog> lateStream = result.getSideOutput(lateDataTag);
        lateStream.addSink(new KafkaSink<>(...)).name("Late-Data-Archive");
    ```

### Conclusion

- As we can see, the notion of time is central to flink's computation. 

- Since we need to manage state within some time bounds, Flink's management of timestamps and watermarks is critical; changing this behaviour can change the result we observe in our app

- We now move on to looking at the actual mechanics of data transformation in flink!